In [1]:
# ==============================================================================
# PIPELINE 8: Regression (Predicting Budget Deviations / Cost Overruns)
# ==============================================================================
import pandas as pd
import numpy as np

from src.training.regression import prepare_regression_data, tune_regression_model
from src.training.evaluation import evaluate_regression
from src.training.visualization import plot_feature_importance, plot_grouped_feature_importance

pd.set_option('display.max_columns', None)

print("Loading preprocessed dataset...")
df_ml = pd.read_parquet("data/prepared/ted_prepared.parquet")

Loading preprocessed dataset...


In [2]:
# ------------------------------------------------------------------------------
# STEP 1: Feature Setup & Data Preparation
# ------------------------------------------------------------------------------

# Note: ESTIMATED_VALUE_EUR is strictly EXCLUDED from features to prevent data leakage,
# as it is used to construct the target variable within the preparation function.
categorical_cols = [
    'ISO_COUNTRY_CODE', 'CAE_TYPE', 'CPV', 
    'TYPE_OF_CONTRACT', 'CRIT_CODE', 'TOP_TYPE'
]

numeric_cols = [
    'DURATION', 'PREPARATION_DAYS', 'LOTS_NUMBER', 
    'NUTS_REGION_COUNT', 'B_EU_FUNDS', 'B_RECURRENT_PROCUREMENT'
]

# The All-in-One function handles level-extraction, ratio calculation, log-transformation, and splitting.
X_train, X_test, y_train, y_test = prepare_regression_data(
    df=df_ml,
    feature_cols=categorical_cols + numeric_cols,
    test_size=0.2
)

Preparing data for Budget Deviation Analysis...
Extraction complete. Notices viable for Budget Analysis: 341,697


/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [3]:
# ------------------------------------------------------------------------------
# STEP 2: Lasso Regression (Linear Baseline)
# ------------------------------------------------------------------------------
model_lasso = tune_regression_model(
    X_train, y_train, categorical_cols, numeric_cols, model_name='lasso'
)

# 1. Predict in the trained Log-Space
preds_lasso_log = model_lasso.predict(X_test)

# 2. Back-transform into real deviation ratios using np.exp
y_test_ratio = np.exp(y_test)
preds_lasso_ratio = np.exp(preds_lasso_log)

# 3. Evaluate using the original function (is_log_transformed=False because we already transformed it)
print("\n--- LASSO REGRESSION ---")
evaluate_regression(y_test_ratio, preds_lasso_ratio, n_features=X_test.shape[1], is_log_transformed=False)

# Plot only if the linear model retained coefficients
if hasattr(model_lasso.named_steps['regressor'], 'coef_'):
    plot_feature_importance(model_lasso, numeric_cols, categorical_cols, top_n=15)

Starting GridSearch for LASSO...


/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in r

ValueError: 
All the 12 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^

In [4]:
# ------------------------------------------------------------------------------
# STEP 3: Random Forest Regressor (Non-linear Baseline)
# ------------------------------------------------------------------------------
model_rf = tune_regression_model(
    X_train, y_train, categorical_cols, numeric_cols, model_name='rf'
)

preds_rf_log = model_rf.predict(X_test)

y_test_ratio = np.exp(y_test)
preds_rf_ratio = np.exp(preds_rf_log)

print("\n--- RANDOM FOREST ---")
evaluate_regression(y_test_ratio, preds_rf_ratio, n_features=X_test.shape[1], is_log_transformed=False)

plot_feature_importance(model_rf, numeric_cols, categorical_cols, top_n=15)
plot_grouped_feature_importance(model_rf, numeric_cols, categorical_cols)

Starting GridSearch for RF...


/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: invalid value encountered in r

ValueError: 
All the 12 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
12 fits failed with the following error:
Traceback (most recent call last):
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^

In [5]:
# ------------------------------------------------------------------------------
# STEP 4: XGBoost Regressor (State of the Art Boosting)
# ------------------------------------------------------------------------------
model_xgb = tune_regression_model(
    X_train, y_train, categorical_cols, numeric_cols, model_name='xgboost'
)

preds_xgb_log = model_xgb.predict(X_test)

y_test_ratio = np.exp(y_test)
preds_xgb_ratio = np.exp(preds_xgb_log)

print("\n--- XGBOOST ---")
evaluate_regression(y_test_ratio, preds_xgb_ratio, n_features=X_test.shape[1], is_log_transformed=False)

plot_feature_importance(model_xgb, numeric_cols, categorical_cols, top_n=15)
plot_grouped_feature_importance(model_xgb, numeric_cols, categorical_cols)

Starting GridSearch for XGBOOST...


ValueError: 
All the 48 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/home/msuess/Documents/KIDS/Project/.venv/lib/python3.13/site-packages/sklearn/pipeline.py", line 621, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^